# CacheBlend All-in-One Colab Notebook

This notebook is self-contained. It includes:
- KV pack/unpack + store
- Adaptive token selector (paper-style thresholds)
- Selective recompute engine
- Blend operation (PyTorch fallback)
- End-to-end CacheBlend pipeline
- Direct smoke tests and a quick TTFT benchmark

Run cells in order.

In [ ]:
# Run once in Colab
!pip install -U pip
!pip install "transformers>=4.45,<5" datasets evaluate rouge_score sentence-transformers bitsandbytes accelerate ninja pytest tqdm

# Optional but recommended for faster HF downloads (set your token in Colab secrets)
# import os
# os.environ["HF_TOKEN"] = "<your_token_here>"

# Verify evaluate loaded (restart kernel and re-run if this errors)
import importlib, evaluate
importlib.reload(evaluate)
print(f'evaluate {evaluate.__version__} ready')


In [ ]:
import json
import time
from statistics import mean, pstdev
from typing import Dict, List, Optional

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


# -----------------------------
# KV helpers + cache store
# -----------------------------
def pack_kv(past_key_values) -> torch.Tensor:
    """HuggingFace cache -> (L, 2, N, H, D)."""
    # DynamicCache-style API
    if hasattr(past_key_values, "key_cache") and hasattr(past_key_values, "value_cache"):
        layers = []
        for k, v in zip(past_key_values.key_cache, past_key_values.value_cache):
            k = k.squeeze(0).transpose(0, 1)  # (N, H, D)
            v = v.squeeze(0).transpose(0, 1)
            layers.append(torch.stack([k, v], dim=0))
        return torch.stack(layers, dim=0)

    # Some cache classes expose a legacy tuple-of-tuples view.
    if hasattr(past_key_values, "to_legacy_cache"):
        past_key_values = past_key_values.to_legacy_cache()

    layers = []
    for layer_kv in past_key_values:
        if not isinstance(layer_kv, (tuple, list)) or len(layer_kv) < 2:
            raise ValueError("Unsupported cache layer format in past_key_values.")

        # Newer HF variants may include extra entries per layer.
        k, v = layer_kv[0], layer_kv[1]
        k = k.squeeze(0).transpose(0, 1)
        v = v.squeeze(0).transpose(0, 1)
        layers.append(torch.stack([k, v], dim=0))

    return torch.stack(layers, dim=0)


def unpack_kv(kv_tensor: torch.Tensor):
    """(L, 2, N, H, D) -> DynamicCache."""
    from transformers import DynamicCache

    cache = DynamicCache()
    for i in range(kv_tensor.shape[0]):
        k = kv_tensor[i, 0].permute(1, 0, 2).unsqueeze(0).half()  # (1, H, N, D)
        v = kv_tensor[i, 1].permute(1, 0, 2).unsqueeze(0).half()
        cache.update(k, v, i)
    return cache


class KVCacheStore:
    def __init__(self):
        self._data = {}

    def store(self, key: str, kv_tensor: torch.Tensor):
        self._data[key] = kv_tensor.cpu().clone()

    def load(self, key: str, device: str = "cuda") -> Optional[torch.Tensor]:
        kv = self._data.get(key)
        if kv is None:
            return None
        return kv.to(device)


# -----------------------------
# Adaptive selector (paper-style)
# -----------------------------
class AdaptiveTokenSelector:
    def __init__(
        self,
        model,
        low_thresh: float = 0.05,
        high_thresh: float = 0.20,
        min_k_ratio: float = 0.05,
        max_k_ratio: float = 0.30,
        eps: float = 1e-8,
    ):
        self.model = model
        self.low_thresh = low_thresh
        self.high_thresh = high_thresh
        self.min_k_ratio = min_k_ratio
        self.max_k_ratio = max_k_ratio
        self.eps = eps
        self.last_selection = {}
        self.history: List[Dict[str, float]] = []
        self.model.eval()

    def _project_cached_for_cosine(self, cached_k_mean: torch.Tensor, target_dim: int) -> torch.Tensor:
        cached_dim = cached_k_mean.shape[-1]
        if cached_dim == target_dim:
            return cached_k_mean
        if cached_dim > target_dim:
            return cached_k_mean[:, :target_dim]
        return F.pad(cached_k_mean, (0, target_dim - cached_dim), mode="constant", value=0.0)

    def _adaptive_ratio(self, mean_divergence: float) -> float:
        if mean_divergence <= self.low_thresh:
            return self.min_k_ratio
        if mean_divergence >= self.high_thresh:
            return self.max_k_ratio
        alpha = (mean_divergence - self.low_thresh) / max(self.high_thresh - self.low_thresh, self.eps)
        return self.min_k_ratio + alpha * (self.max_k_ratio - self.min_k_ratio)

    def select(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor) -> torch.Tensor:
        # We have no padding here, so an all-ones mask is appropriate.
        attn_mask = torch.ones_like(chunk_ids, device=chunk_ids.device)

        with torch.no_grad():
            out = self.model(
                chunk_ids,
                attention_mask=attn_mask,
                use_cache=True,
                output_hidden_states=True,
                return_dict=True,
            )
            fresh_hidden = out.hidden_states[1].squeeze(0).float()  # (N, d_model)

        cached_k = cached_kv[0, 0].float()  # (N, H, D)
        cached_hidden = cached_k.mean(dim=1)  # (N, D)
        cached_hidden = self._project_cached_for_cosine(cached_hidden, fresh_hidden.shape[-1])

        fresh_norm = F.normalize(fresh_hidden + self.eps, p=2, dim=-1)
        cached_norm = F.normalize(cached_hidden + self.eps, p=2, dim=-1)

        cosine = F.cosine_similarity(fresh_norm, cached_norm, dim=-1)
        divergence = (1.0 - cosine).clamp_min(0.0)

        mean_div = float(divergence.mean().item())
        ratio = self._adaptive_ratio(mean_div)

        n_tokens = int(chunk_ids.shape[1])
        k = min(n_tokens, max(1, int(ratio * n_tokens)))
        indices = torch.topk(divergence, k, largest=True, sorted=False).indices
        indices = torch.sort(indices).values.to(torch.int64)

        self.last_selection = {
            "sequence_length": float(n_tokens),
            "mean_divergence": mean_div,
            "selected_k_ratio": float(ratio),
            "selected_k": float(k),
        }
        self.history.append(self.last_selection.copy())

        return indices


# -----------------------------
# Recompute + blend + pipeline
# -----------------------------
class SelectiveRecomputer:
    def __init__(self, model):
        self.model = model

    def recompute(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        selected_ids = chunk_ids[:, indices]  # (1, k)
        past = unpack_kv(cached_kv)

        with torch.no_grad():
            out = self.model(selected_ids, past_key_values=past, use_cache=True)

        new_kv_full = pack_kv(out.past_key_values)  # (L, 2, N+k, H, D)
        k = indices.shape[0]
        return new_kv_full[:, :, -k:, :, :]


class KVBlender:
    def blend(self, cached_kv: torch.Tensor, new_values: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        cached_kv[:, :, indices, :, :] = new_values
        return cached_kv


def cacheblend_generate(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    store,
    selector,
    recomputer,
    blender=None,
    mode: str = "cacheblend",  # "standard_cache" or "cacheblend"
    max_new_tokens=32,
    min_new_tokens=8,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    benchmark_first_token: bool = False,
):
    if blender is None:
        blender = KVBlender()

    cache_hits, cache_misses = 0, 0
    fused_kv = None

    # End-to-end TTFT should include chunk processing + first-token generation.
    torch.cuda.synchronize()
    t_request_start = time.perf_counter()

    for chunk_text in chunk_texts:
        chunk_enc = tokenizer(
            chunk_text,
            return_tensors="pt",
            return_attention_mask=True,
            truncation=True,
            max_length=1024,
        )
        chunk_ids = chunk_enc["input_ids"].to("cuda")
        chunk_attn = chunk_enc["attention_mask"].to("cuda")

        cached = store.load(chunk_text, device="cuda")

        if cached is None:
            cache_misses += 1
            with torch.no_grad():
                out = model(chunk_ids, attention_mask=chunk_attn, use_cache=True)
            chunk_kv = pack_kv(out.past_key_values)
            store.store(chunk_text, chunk_kv)
        else:
            cache_hits += 1
            chunk_kv = cached

            if mode == "cacheblend":
                indices = selector.select(chunk_ids, chunk_kv)
                new_kv = recomputer.recompute(chunk_ids, chunk_kv, indices)
                chunk_kv = blender.blend(chunk_kv, new_kv, indices)
            elif mode == "standard_cache":
                # Reuse cached KV directly.
                pass
            else:
                raise ValueError(f"Unsupported mode: {mode}")

        fused_kv = chunk_kv if fused_kv is None else torch.cat([fused_kv, chunk_kv], dim=2)

    prompt_enc = tokenizer(
        prompt,
        return_tensors="pt",
        return_attention_mask=True,
        truncation=True,
        max_length=128,
    )
    prompt_ids = prompt_enc["input_ids"].to("cuda")
    prompt_attn = prompt_enc["attention_mask"].to("cuda")

    past = unpack_kv(fused_kv) if fused_kv is not None else None

    torch.cuda.synchronize()
    t_decode_start = time.perf_counter()

    context_len = fused_kv.shape[2] if fused_kv is not None else 0
    cache_position = torch.arange(context_len, context_len + prompt_ids.shape[1], device=prompt_ids.device)

    gen_max = 1 if benchmark_first_token else max_new_tokens
    gen_min = 1 if benchmark_first_token else min_new_tokens
    gen_sample = False if benchmark_first_token else do_sample

    with torch.no_grad():
        out_ids = model.generate(
            prompt_ids,
            attention_mask=prompt_attn,
            past_key_values=past,
            cache_position=cache_position,
            max_new_tokens=gen_max,
            min_new_tokens=gen_min,
            use_cache=True,
            do_sample=gen_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    decode_only_ms = (time.perf_counter() - t_decode_start) * 1000
    e2e_ttft_ms = (time.perf_counter() - t_request_start) * 1000

    generated = out_ids[0][prompt_ids.shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "text": text,
        "ttft_ms": e2e_ttft_ms,
        "e2e_ttft_ms": e2e_ttft_ms,
        "generate_only_ms": decode_only_ms,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
    }


def run_three_mode_benchmark(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    trials: int = 5,
    warmup_runs: int = 1,
):
    cold_full = []
    warm_standard = []
    warm_cacheblend = []
    per_trial = []

    for _ in range(warmup_runs):
        s = KVCacheStore()
        sel = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec = SelectiveRecomputer(model)
        bl = KVBlender()
        _ = cacheblend_generate(prompt, chunk_texts, model, tokenizer, s, sel, rec, bl, mode="standard_cache", benchmark_first_token=True)
        _ = cacheblend_generate(prompt, chunk_texts, model, tokenizer, s, sel, rec, bl, mode="cacheblend", benchmark_first_token=True)

    for t in range(trials):
        # 1) Cold full prefill request
        store_cold = KVCacheStore()
        sel_cold = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_cold = SelectiveRecomputer(model)
        bl_cold = KVBlender()
        cold = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cold,
            sel_cold,
            rec_cold,
            bl_cold,
            mode="standard_cache",
            benchmark_first_token=True,
        )

        # 2) Warm standard KV cache request (no recompute)
        store_std = KVCacheStore()
        sel_std = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_std = SelectiveRecomputer(model)
        bl_std = KVBlender()
        _ = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_std,
            sel_std,
            rec_std,
            bl_std,
            mode="standard_cache",
            benchmark_first_token=True,
        )
        warm_std = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_std,
            sel_std,
            rec_std,
            bl_std,
            mode="standard_cache",
            benchmark_first_token=True,
        )

        # 3) Warm CacheBlend request (adaptive recompute + blend)
        store_cb = KVCacheStore()
        sel_cb = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_cb = SelectiveRecomputer(model)
        bl_cb = KVBlender()
        _ = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=True,
        )
        warm_cb = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=True,
        )

        cold_full.append(cold["e2e_ttft_ms"])
        warm_standard.append(warm_std["e2e_ttft_ms"])
        warm_cacheblend.append(warm_cb["e2e_ttft_ms"])

        per_trial.append(
            {
                "trial": t + 1,
                "cold_full_ttft_ms": cold["e2e_ttft_ms"],
                "warm_standard_ttft_ms": warm_std["e2e_ttft_ms"],
                "warm_cacheblend_ttft_ms": warm_cb["e2e_ttft_ms"],
                "speedup_cold_vs_standard": cold["e2e_ttft_ms"] / max(warm_std["e2e_ttft_ms"], 1e-9),
                "speedup_cold_vs_cacheblend": cold["e2e_ttft_ms"] / max(warm_cb["e2e_ttft_ms"], 1e-9),
                "cacheblend_selector_stats": sel_cb.last_selection,
            }
        )

    summary = {
        "trials": trials,
        "warmup_runs": warmup_runs,
        "cold_full_mean_ms": mean(cold_full),
        "cold_full_std_ms": pstdev(cold_full) if len(cold_full) > 1 else 0.0,
        "warm_standard_mean_ms": mean(warm_standard),
        "warm_standard_std_ms": pstdev(warm_standard) if len(warm_standard) > 1 else 0.0,
        "warm_cacheblend_mean_ms": mean(warm_cacheblend),
        "warm_cacheblend_std_ms": pstdev(warm_cacheblend) if len(warm_cacheblend) > 1 else 0.0,
        "speedup_cold_vs_standard_mean": mean([c / max(s, 1e-9) for c, s in zip(cold_full, warm_standard)]),
        "speedup_cold_vs_cacheblend_mean": mean([c / max(cb, 1e-9) for c, cb in zip(cold_full, warm_cacheblend)]),
    }

    return {"summary": summary, "trials": per_trial}


print("Definitions loaded.")

In [ ]:
# Runtime + model load
assert torch.cuda.is_available(), "Please enable GPU runtime in Colab (T4)."

device = "cuda"
# Use GPT-2 for cleaner behavior than tiny-gpt2 while still fitting Colab T4 comfortably.
model_id = "gpt2"

print(f"Loading: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to(device).eval()

print("Model loaded.")

In [ ]:
# Direct Colab smoke tests + repeated benchmark (no pytest needed)

# Test 1: adaptive selector output contract
text = (
    "CacheBlend recomputes only highly divergent tokens to reduce prefill latency. "
    "This validates adaptive token selection behavior."
)
chunk_enc = tokenizer(text, return_tensors="pt", return_attention_mask=True)
chunk_ids = chunk_enc["input_ids"].to(device)
chunk_mask = chunk_enc["attention_mask"].to(device)

with torch.no_grad():
    out = model(chunk_ids, attention_mask=chunk_mask, use_cache=True)

cached_kv = pack_kv(out.past_key_values).to(device)
selector = AdaptiveTokenSelector(model=model)
indices = selector.select(chunk_ids, cached_kv)
stats = selector.last_selection

assert indices.dtype == torch.int64
assert indices.device.type == "cuda"
assert indices.numel() >= 1
assert torch.all(indices[1:] >= indices[:-1]), "Indices must be sorted"
assert 0.05 <= stats["selected_k_ratio"] <= 0.30
print("Test 1 passed:", stats)


# Test 2: end-to-end cold/warm run
store = KVCacheStore()
recomputer = SelectiveRecomputer(model)
blender = KVBlender()

chunks = [
    "Paris is the capital of France and has many famous landmarks. " * 8,
    "The Eiffel Tower is one of the most visited monuments in the world. " * 8,
    "CacheBlend reuses cached KV and selectively recomputes high-divergence tokens. " * 8,
    "Selective recomputation reduces full prefill cost when cached context is reusable. " * 8,
]
prompt = "Summarize how CacheBlend can reduce latency for repeated context queries."

cold = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)
warm = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)

assert cold["cache_misses"] == len(chunks)
assert warm["cache_hits"] == len(chunks)
assert len(warm["text"]) > 0
assert warm["ttft_ms"] > 0

print("Test 2 passed")
print(f"COLD TTFT: {cold['ttft_ms']:.2f} ms")
print(f"WARM TTFT: {warm['ttft_ms']:.2f} ms")
print(f"Speedup: {cold['ttft_ms']/warm['ttft_ms']:.2f}x")
print("Output:", warm["text"][:200])
print("Output repr:", repr(warm["text"][:200]))


# Test 3: repeated benchmark for stable report numbers
bench = run_three_mode_benchmark(
    prompt=prompt,
    chunk_texts=chunks,
    model=model,
    tokenizer=tokenizer,
    trials=5,
    warmup_runs=1,
)

print("\nBenchmark summary:")
for k, v in bench["summary"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

with open("cacheblend_colab_results.json", "w") as f:
    json.dump(bench, f, indent=2)

print("Saved benchmark JSON -> cacheblend_colab_results.json")

## Phase 2 — Full Eval Harness (Mistral-7B · MultiNews)

Produces the three tables from the CacheBlend+ report:

| Table | Metric | Configs |
|-------|--------|---------|
| 1 | TTFT (ms) | k/N ∈ {5 %, 10 %, 15 %, 20 %, 30 %} vs full-prefill |
| 2 | ROUGE-L | same k/N sweep |
| 3 | ROUGE-L grid | {fixed, adaptive} × {exact, semantic} |

> **GPU required.** Run on a Colab A100 or T4 instance.  
> Results are saved incrementally to `/content/drive/MyDrive/cacheblend_results/`  
> so the notebook is safe to preempt and resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/cacheblend_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')


In [ ]:
# ── Eval-harness helpers (all-inline; no package install needed) ────────────
import json, time
from pathlib import Path
from statistics import mean


def load_rouge():
    try:
        import evaluate
        return evaluate.load("rouge")
    except Exception:
        print("WARNING: 'evaluate' not installed.")
        return None


def load_multinews(n_samples: int) -> list:
    from datasets import load_dataset
    try:
        ds = load_dataset("multi_news", split=f"validation[:{n_samples}]",
                          trust_remote_code=True)
        samples = []
        for ex in ds:
            articles = [a.strip() for a in ex["document"].split("|||||") if a.strip()]
            summary  = ex["summary"].strip()
            if articles and summary:
                samples.append({"chunks": articles[:4], "summary": summary})
        return samples[:n_samples]
    except Exception as e:
        print(f"WARNING: MultiNews load failed ({e}) — using synthetic fallback")
        return [{
            "chunks": [
                "Scientists discovered a new deep-sea fish in the Pacific Ocean. "
                "The fish lives at depths over 3000 m and has bioluminescent properties.",
                "The discovery adds to a growing list of deep-sea species. "
                "Researchers named it Abyssus luminis.",
            ],
            "summary": "Researchers found a new bioluminescent deep-sea fish in the Pacific.",
        }] * min(n_samples, 5)


def measure_prefill_ttft(model, input_ids, n_runs=3):
    # Median wall-clock prefill time (ms) over n_runs trials.
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            model(input_ids, use_cache=True)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[n_runs // 2]


def _standard_cache_generate(model, tokenizer, chunks, max_new_tokens=128):
    # Baseline generation: full prefill, no CacheBlend.
    store_tmp = KVCacheStore()
    dummy_sel = TokenSelector(k_ratio=0.15)   # never used (cold pass only)
    r = cacheblend_generate(
        "Summarize the above in 2-3 sentences:",
        chunks, model, tokenizer,
        store_tmp, dummy_sel,
        SelectiveRecomputer(model), KVBlender(),
        mode="standard_cache",
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return r["text"]


class TokenSelector:
    # Fixed k/N token selector (evenly spaced positions).
    def __init__(self, k_ratio=0.15):
        self.k_ratio = float(k_ratio)

    def select(self, chunk_ids, cached_kv):
        N = int(chunk_ids.shape[1])
        k = max(1, min(N, int(self.k_ratio * N)))
        indices = torch.linspace(0, N - 1, k, dtype=torch.int64, device=chunk_ids.device)
        return torch.sort(indices).values


def eval_table1_table2(model, tokenizer, samples, recompute_ratios,
                       recomputer, blender, results_dir, max_new_tokens=128):
    outpath = Path(results_dir) / "table1_table2.json"
    results = {}
    if outpath.exists():
        with open(outpath) as f:
            results = json.load(f)
        print(f"Resuming table1_table2 ({len(results)} entries already done)")

    rouge = load_rouge()

    # ── Baseline: full prefill ─────────────────────────────────────────────
    if "baseline" not in results:
        print("\n[Baseline] full-prefill TTFT + ROUGE-L ...")
        base_ttfts, base_preds = [], []
        for i, s in enumerate(samples):
            all_text = " ".join(s["chunks"])
            ids = tokenizer(all_text, return_tensors="pt",
                            truncation=True, max_length=2048)["input_ids"].cuda()
            base_ttfts.append(measure_prefill_ttft(model, ids))
            base_preds.append(
                _standard_cache_generate(model, tokenizer, s["chunks"], max_new_tokens)
            )
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        base_rouge = None
        if rouge:
            refs = [s["summary"] for s in samples]
            base_rouge = rouge.compute(predictions=base_preds, references=refs)["rougeL"]

        results["baseline"] = {
            "mean_ttft_ms": mean(base_ttfts),
            "rougeL": base_rouge,
        }
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        rl = results["baseline"]["rougeL"]
        print(f"  Baseline TTFT={results['baseline']['mean_ttft_ms']:.1f} ms  "
              f"ROUGE-L={f'{rl:.4f}' if rl else 'N/A'}")

    # ── CacheBlend sweep ────────────────────────────────────────────────────
    for ratio in recompute_ratios:
        key = f"ratio_{ratio:.2f}"
        if key in results:
            print(f"  Skipping ratio={ratio} (cached)")
            continue

        print(f"\n[CacheBlend] k/N={ratio:.0%} ...")
        store    = KVCacheStore()
        selector = TokenSelector(k_ratio=ratio)
        cb_ttfts, cb_preds = [], []

        for i, s in enumerate(samples):
            chunks = s["chunks"]
            prompt = "Summarize the above in 2-3 sentences:"

            # Cold pass — populate cache
            cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=1, do_sample=False)

            # TTFT proxy: time prefill on k tokens (matches paper's Table 1)
            all_text = " ".join(chunks)
            ids = tokenizer(all_text, return_tensors="pt",
                            truncation=True, max_length=2048)["input_ids"].cuda()
            k = max(1, int(ratio * ids.shape[1]))
            cb_ttfts.append(measure_prefill_ttft(model, ids[:, :k]))

            # Warm pass — quality measurement
            r = cacheblend_generate(prompt, chunks, model, tokenizer,
                                    store, selector, recomputer, blender,
                                    max_new_tokens=max_new_tokens, do_sample=False)
            cb_preds.append(r["text"])
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        cb_rouge = None
        if rouge:
            refs = [s["summary"] for s in samples]
            cb_rouge = rouge.compute(predictions=cb_preds, references=refs)["rougeL"]

        mean_ttft = mean(cb_ttfts)
        results[key] = {
            "ratio": ratio,
            "mean_ttft_ms": mean_ttft,
            "speedup_vs_baseline": results["baseline"]["mean_ttft_ms"] / mean_ttft,
            "rougeL": cb_rouge,
        }
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        rl = cb_rouge
        print(f"  ratio={ratio:.0%}  TTFT={mean_ttft:.1f} ms  "
              f"speedup={results[key]['speedup_vs_baseline']:.2f}x  "
              f"ROUGE-L={f'{rl:.4f}' if rl else 'N/A'}")

    return results


def eval_table3(model, tokenizer, samples,
                exact_store_class, semantic_store_class,
                recomputer, blender,
                results_dir, k_ratio=0.15, max_new_tokens=128):
    outpath = Path(results_dir) / "table3.json"
    results = {}
    if outpath.exists():
        with open(outpath) as f:
            results = json.load(f)

    rouge = load_rouge()

    fixed_factory    = lambda k_ratio: TokenSelector(k_ratio=k_ratio)
    adaptive_factory = lambda k_ratio: AdaptiveTokenSelector(
        model=model,
        low_thresh=0.05, high_thresh=0.20,
        min_k_ratio=max(0.05, k_ratio * 0.5),
        max_k_ratio=min(k_ratio * 2, 0.50),
    )

    configs = [
        ("fixed",    "exact",    fixed_factory,    exact_store_class),
        ("adaptive", "exact",    adaptive_factory, exact_store_class),
    ]
    if semantic_store_class is not None:
        configs += [
            ("fixed",    "semantic", fixed_factory,    semantic_store_class),
            ("adaptive", "semantic", adaptive_factory, semantic_store_class),
        ]

    for sel_type, store_type, sel_fn, store_cls in configs:
        key = f"{sel_type}_{store_type}"
        if key in results:
            print(f"  Skipping {key} (cached)")
            continue

        print(f"\n[Table 3] {sel_type} x {store_type} ...")
        store    = store_cls()
        selector = sel_fn(k_ratio=k_ratio)
        preds    = []

        for i, s in enumerate(samples):
            chunks = s["chunks"]
            prompt = "Summarize the above in 2-3 sentences:"
            cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=1, do_sample=False)
            r = cacheblend_generate(prompt, chunks, model, tokenizer,
                                    store, selector, recomputer, blender,
                                    max_new_tokens=max_new_tokens, do_sample=False)
            preds.append(r["text"])
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        score = None
        if rouge:
            refs = [s["summary"] for s in samples]
            score = rouge.compute(predictions=preds, references=refs)["rougeL"]

        results[key] = {"selector": sel_type, "store": store_type, "rougeL": score}
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        print(f"  {key}: ROUGE-L={f'{score:.4f}' if score else 'N/A'}")

    return results


def print_tables(t12, t3):
    print("\n" + "=" * 65)
    print("TABLE 1 & 2  |  MultiNews  |  Mistral-7B-Instruct-v0.2")
    print("=" * 65)
    b  = t12.get("baseline", {})
    bl = b.get("rougeL")
    print(f"{'Config':<24} {'TTFT (ms)':>10} {'Speedup':>10} {'ROUGE-L':>10}")
    print("-" * 56)
    print(f"{'Full prefill':<24} {b.get('mean_ttft_ms', 0):>10.1f} "
          f"{'1.00x':>10} {f'{bl:.4f}' if bl else 'N/A':>10}")
    for k in sorted(t12):
        if not k.startswith("ratio_"):
            continue
        v  = t12[k]
        rl = v.get("rougeL")
        label = f"CacheBlend {int(v['ratio']*100)}%"
        print(f"{label:<24} {v['mean_ttft_ms']:>10.1f} "
              f"{v['speedup_vs_baseline']:>9.2f}x "
              f"{f'{rl:.4f}' if rl else 'N/A':>10}")

    if t3:
        print("\n" + "=" * 48)
        print("TABLE 3  |  ROUGE-L  |  selector x cache store")
        print("=" * 48)
        print(f"{'':>16} {'Exact':>14} {'Semantic':>14}")
        print("-" * 46)
        for sel in ["fixed", "adaptive"]:
            exact    = t3.get(f"{sel}_exact",    {}).get("rougeL")
            semantic = t3.get(f"{sel}_semantic", {}).get("rougeL")
            print(f"{sel.capitalize():<16} "
                  f"{f'{exact:.4f}' if exact else 'N/A':>14} "
                  f"{f'{semantic:.4f}' if semantic else 'N/A':>14}")


print("Eval-harness helpers defined.")


In [ ]:
# Load Mistral-7B-Instruct in 8-bit (fits on T4/A100)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MISTRAL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
print(f'Loading {MISTRAL_ID} in 8-bit ...')

bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
mistral_tok = AutoTokenizer.from_pretrained(MISTRAL_ID)
mistral_tok.pad_token = mistral_tok.eos_token
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_ID,
    quantization_config=bnb_cfg,
    device_map='cuda',
).eval()

mistral_recomputer = SelectiveRecomputer(mistral_model)
mistral_blender    = KVBlender()
print('Mistral-7B ready.')


In [ ]:
N_SAMPLES        = 60
MAX_NEW_TOKENS   = 128
RECOMPUTE_RATIOS = [0.05, 0.10, 0.15, 0.20, 0.30]

print(f'Loading {N_SAMPLES} MultiNews samples ...')
mn_samples = load_multinews(N_SAMPLES)
print(f'  Loaded {len(mn_samples)} samples')

t12_results = eval_table1_table2(
    mistral_model, mistral_tok, mn_samples,
    RECOMPUTE_RATIOS,
    mistral_recomputer, mistral_blender,
    RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
)


In [ ]:
# Sequence-length diagnostic — run after mn_samples is loaded
# Shows actual token counts so we can judge whether context is long enough
# for meaningful TTFT speedups (paper uses 2k-8k token contexts).
_lens = [
    len(mistral_tok(' '.join(s['chunks']), truncation=True,
                    max_length=4096)['input_ids'])
    for s in mn_samples
]
_sorted = sorted(_lens)
print(f'Sequence lengths across {len(_lens)} samples:')
print(f'  Mean : {sum(_lens)/len(_lens):.0f} tokens')
print(f'  Min  : {_sorted[0]} tokens')
print(f'  p25  : {_sorted[len(_sorted)//4]} tokens')
print(f'  p50  : {_sorted[len(_sorted)//2]} tokens')
print(f'  p75  : {_sorted[3*len(_sorted)//4]} tokens')
print(f'  Max  : {_sorted[-1]} tokens')
if sum(_lens)/len(_lens) < 600:
    print('NOTE: Mean sequence is short (<600 tokens). '
          'Speedups will be modest. Consider using more chunks per sample.')


In [ ]:
# sentence-transformers needed for semantic store (Table 3 col 2).
# Set USE_SEMANTIC=False to skip it.
USE_SEMANTIC = True

SemanticStore = None
if USE_SEMANTIC:
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as _np

        class SemanticKVCacheStore(KVCacheStore):
            def __init__(self, model_name='all-MiniLM-L6-v2', threshold=0.85):
                super().__init__()
                self.encoder   = SentenceTransformer(model_name, device='cpu')
                self.threshold = threshold
                self._embeddings = []

            def store(self, text, kv_tensor):
                self._data[text] = kv_tensor.cpu().clone()
                emb = self.encoder.encode(text, convert_to_numpy=True)
                self._embeddings.append((emb, text))

            def load(self, text, device='cuda'):
                kv = self._data.get(text)
                if kv is not None:
                    return kv.to(device)
                if not self._embeddings:
                    return None
                q = self.encoder.encode(text, convert_to_numpy=True)
                best_sim, best_key = -1.0, None
                for emb, key in self._embeddings:
                    sim = float(_np.dot(q, emb) /
                                (_np.linalg.norm(q) * _np.linalg.norm(emb) + 1e-8))
                    if sim > best_sim:
                        best_sim, best_key = sim, key
                if best_sim >= self.threshold and best_key in self._data:
                    return self._data[best_key].to(device)
                return None

        SemanticStore = SemanticKVCacheStore
        print('SemanticKVCacheStore ready.')
    except ImportError:
        print('sentence-transformers not installed -- semantic column will be N/A')

t3_results = eval_table3(
    mistral_model, mistral_tok, mn_samples,
    KVCacheStore, SemanticStore,
    mistral_recomputer, mistral_blender,
    RESULTS_DIR, k_ratio=0.15,
    max_new_tokens=MAX_NEW_TOKENS,
)


### ROUGE-L Repair

The eval ran successfully but `evaluate` wasn't importable during that session,
so all `rougeL` fields are `null`. This cell re-generates predictions for every
null entry (skipping TTFT re-measurement) and fills in the scores.

In [ ]:
# ROUGE-L Repair: fills null rougeL without re-measuring TTFT
import json, importlib
from pathlib import Path

# Force-reload evaluate in case it was installed mid-session
try:
    import evaluate as _ev
    importlib.reload(_ev)
    rouge = _ev.load("rouge")
    print(f"evaluate {_ev.__version__} ready")
except Exception as e:
    raise RuntimeError(
        f"evaluate not importable: {e}\n"
        "Run:  !pip install evaluate rouge_score -q\n"
        "then Restart Runtime and re-run from the top."
    )


def _generate_preds(model, tokenizer, samples, store_class, sel_fn,
                    recomputer, blender, max_new_tokens=128):
    # Re-run warm cacheblend_generate for each sample; return list of prediction strings.
    preds = []
    for i, s in enumerate(samples):
        chunks = s["chunks"]
        prompt = "Summarize the above in 2-3 sentences:"
        store    = store_class()
        selector = sel_fn()
        # Cold pass to populate cache
        cacheblend_generate(prompt, chunks, model, tokenizer,
                            store, selector, recomputer, blender,
                            max_new_tokens=1, do_sample=False)
        # Warm pass for quality
        r = cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=max_new_tokens, do_sample=False)
        preds.append(r["text"])
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(samples)}")
    return preds


def repair_rouge(results_dir, model, tokenizer, samples,
                 recomputer, blender, max_new_tokens=128):
    refs = [s["summary"] for s in samples]

    # ── table1_table2.json ────────────────────────────────────────────────
    t12_path = Path(results_dir) / "table1_table2.json"
    with open(t12_path) as f:
        t12 = json.load(f)

    t12_dirty = False

    # Baseline — uses standard_cache (no selector needed)
    if t12.get("baseline", {}).get("rougeL") is None and "baseline" in t12:
        print("\n[Repair] baseline ROUGE-L ...")
        preds = _generate_preds(
            model, tokenizer, samples,
            KVCacheStore,
            lambda: TokenSelector(k_ratio=1.0),   # k=100% => standard cache behaviour
            recomputer, blender, max_new_tokens,
        )
        t12["baseline"]["rougeL"] = rouge.compute(
            predictions=preds, references=refs
        )["rougeL"]
        t12_dirty = True
        print(f"  baseline rougeL = {t12['baseline']['rougeL']:.4f}")

    for key, entry in t12.items():
        if not key.startswith("ratio_") or entry.get("rougeL") is not None:
            continue
        ratio = entry["ratio"]
        print(f"\n[Repair] ratio={ratio:.0%} ROUGE-L ...")
        preds = _generate_preds(
            model, tokenizer, samples,
            KVCacheStore,
            lambda r=ratio: TokenSelector(k_ratio=r),
            recomputer, blender, max_new_tokens,
        )
        entry["rougeL"] = rouge.compute(
            predictions=preds, references=refs
        )["rougeL"]
        t12_dirty = True
        print(f"  ratio={ratio:.0%} rougeL = {entry['rougeL']:.4f}")

    if t12_dirty:
        with open(t12_path, "w") as f:
            json.dump(t12, f, indent=2)
        print(f"\nSaved updated {t12_path}")

    # ── table3.json ────────────────────────────────────────────────────────
    t3_path = Path(results_dir) / "table3.json"
    if not t3_path.exists():
        print("table3.json not found — skipping")
        return t12, {}

    with open(t3_path) as f:
        t3 = json.load(f)

    t3_dirty = False
    sel_map = {
        "fixed":    lambda: TokenSelector(k_ratio=0.15),
        "adaptive": lambda: AdaptiveTokenSelector(
            model=model,
            low_thresh=0.05, high_thresh=0.20,
            min_k_ratio=0.075, max_k_ratio=0.30,
        ),
    }

    for key, entry in t3.items():
        if entry.get("rougeL") is not None:
            continue
        sel_type   = entry["selector"]
        store_type = entry["store"]
        print(f"\n[Repair] Table 3  {sel_type} x {store_type} ...")

        if store_type == "semantic":
            try:
                from sentence_transformers import SentenceTransformer
                import numpy as _np

                class _SemanticStore(KVCacheStore):
                    def __init__(self):
                        super().__init__()
                        self.encoder   = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
                        self.threshold = 0.85
                        self._embeddings = []
                    def store(self, text, kv):
                        self._data[text] = kv.cpu().clone()
                        self._embeddings.append(
                            (self.encoder.encode(text, convert_to_numpy=True), text)
                        )
                    def load(self, text, device="cuda"):
                        kv = self._data.get(text)
                        if kv is not None:
                            return kv.to(device)
                        if not self._embeddings:
                            return None
                        q = self.encoder.encode(text, convert_to_numpy=True)
                        best_sim, best_key = -1.0, None
                        for emb, k in self._embeddings:
                            sim = float(_np.dot(q, emb) /
                                        (_np.linalg.norm(q)*_np.linalg.norm(emb)+1e-8))
                            if sim > best_sim:
                                best_sim, best_key = sim, k
                        if best_sim >= self.threshold and best_key in self._data:
                            return self._data[best_key].to(device)
                        return None

                store_cls = _SemanticStore
            except ImportError:
                print("  sentence-transformers not installed — skipping semantic entry")
                continue
        else:
            store_cls = KVCacheStore

        preds = _generate_preds(
            model, tokenizer, samples,
            store_cls,
            sel_map[sel_type],
            recomputer, blender, max_new_tokens,
        )
        entry["rougeL"] = rouge.compute(predictions=preds, references=refs)["rougeL"]
        t3_dirty = True
        print(f"  {key} rougeL = {entry['rougeL']:.4f}")

    if t3_dirty:
        with open(t3_path, "w") as f:
            json.dump(t3, f, indent=2)
        print(f"\nSaved updated {t3_path}")

    return t12, t3


# ── Run the repair ─────────────────────────────────────────────────────────
t12_results, t3_results = repair_rouge(
    RESULTS_DIR,
    mistral_model, mistral_tok, mn_samples,
    mistral_recomputer, mistral_blender,
    max_new_tokens=128,
)

print_tables(t12_results, t3_results)


In [ ]:
print_tables(t12_results, t3_results)

summary = {'table1_table2': t12_results, 'table3': t3_results}
summary_path = f'{RESULTS_DIR}/eval_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nFull results saved to {RESULTS_DIR}/')
print(f'  table1_table2.json  -- per-ratio TTFT + ROUGE-L')
print(f'  table3.json         -- selector x store ROUGE-L grid')
print(f'  eval_summary.json   -- combined summary')
